In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install numpy pandas scikit-learn==1.7.2 imbalanced-learn xgboost lightgbm tensorflow joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 25.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/eCard_Validator/data/creditcard.csv"

df = pd.read_csv(DATA_PATH)

X = df.drop("Class", axis=1)
y = df["Class"]

print("Data loaded. Shape:", df.shape)
print("Number of positive cases:", y.sum())

Data loaded. Shape: (284807, 31)
Number of positive cases: 492


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Features scaled. Shape:", X_scaled.shape)

Features scaled. Shape: (284807, 30)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (227845, 30)
Test shape: (56962, 30)


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print("Class weights:", class_weight_dict)

Class weights: {0: np.float64(0.5008661206149896), 1: np.float64(289.14340101522845)}


In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import lightgbm as lgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

meta_X = np.zeros((X_train.shape[0], 3))
meta_y = np.zeros(X_train.shape[0])

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"Training fold {fold}...")

    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    smote = SMOTE(random_state=42)
    X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)

    xgb_fold = XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        scale_pos_weight=class_weight_dict[1], eval_metric="logloss"
    )
    xgb_fold.fit(X_tr_res, y_tr_res)

    lgbm_fold = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=7, class_weight="balanced"
    )
    lgbm_fold.fit(X_tr_res, y_tr_res)

    dnn_fold = Sequential([
        Dense(64, activation='relu', input_shape=(X_tr_res.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    dnn_fold.compile(optimizer=Adam(0.001), loss="binary_crossentropy")

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    dnn_fold.fit(
        X_tr_res, y_tr_res,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=256,
        verbose=0,
        callbacks=[early_stop]
    )

    meta_X[val_idx, :] = np.column_stack((
        xgb_fold.predict_proba(X_val)[:, 1],
        lgbm_fold.predict_proba(X_val)[:, 1],
        dnn_fold.predict(X_val).flatten()
    ))
    meta_y[val_idx] = y_val

print("Stacking OOF predictions ready.")

Training fold 1...
[LightGBM] [Info] Number of positive: 181960, number of negative: 181960
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.201286 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 363920, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1425/1425 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Training fold 2...
[LightGBM] [Info] Number of positive: 181961, number of negative: 181961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.126637 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 363922, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1425/1425 ━━━━━━━━━━━━━━━━━━━━ 1s 953us/step
Training fold 3...
[LightGBM] [Info] Number of positive: 181961, number of negative: 181961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.137743 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 363922, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


...

1425/1425 ━━━━━━━━━━━━━━━━━━━━ 1s 963us/step
Training fold 4...
[LightGBM] [Info] Number of positive: 181961, number of negative: 181961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.122907 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 363922, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1425/1425 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Training fold 5...
[LightGBM] [Info] Number of positive: 181961, number of negative: 181961
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.126235 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 363922, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1425/1425 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Stacking OOF predictions ready.


In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(class_weight='balanced')
meta_model.fit(meta_X, meta_y)

print("Meta-model trained.")

Meta-model trained.


In [ ]:
from xgboost import XGBClassifier
import lightgbm as lgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import joblib

smote_full = SMOTE(random_state=42)
X_train_res, y_train_res = smote_full.fit_resample(X_train, y_train)

xgb = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    scale_pos_weight=class_weight_dict[1], eval_metric="logloss"
)
xgb.fit(X_train_res, y_train_res)
joblib.dump(xgb, "/content/drive/MyDrive/eCard_Validator/models2/xgb_model.pkl")

lgbm = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=7, class_weight="balanced"
)
lgbm.fit(X_train_res, y_train_res)
joblib.dump(lgbm, "/content/drive/MyDrive/eCard_Validator/models2/lgbm_model.pkl")

dnn = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_res.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
dnn.compile(optimizer=Adam(0.001), loss="binary_crossentropy", metrics=["accuracy"])

early_stop_full = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
dnn.fit(
    X_train_res, y_train_res,
    validation_split=0.1,
    epochs=50,
    batch_size=256,
    verbose=1,
    callbacks=[early_stop_full]
)
dnn.save("/content/drive/MyDrive/eCard_Validator/models2/dnn_model.h5")

joblib.dump(meta_model, "/content/drive/MyDrive/eCard_Validator/models2/meta_model.pkl")

print("Base models retrained on full data and saved.")

[LightGBM] [Info] Number of positive: 227451, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.145207 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 454902, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1600/1600 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9367 - loss: 0.1469 - val_accuracy: 0.9988 - val_loss: 0.0167
Epoch 2/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9930 - loss: 0.0235 - val_accuracy: 1.0000 - val_loss: 0.0036
Epoch 3/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9968 - loss: 0.0117 - val_accuracy: 0.9997 - val_loss: 0.0036
Epoch 4/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9982 - loss: 0.0077 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 5/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9986 - loss: 0.0062 - val_accuracy: 1.0000 - val_loss: 8.5472e-04
Epoch 6/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9987 - loss: 0.0058 - val_accuracy: 1.0000 - val_loss: 7.2135e-04
Epoch 7/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9989 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 0.0010
Epoch 8/50
1600/1600 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.9990 - loss: 0.0046 - v

Base models retrained on full data and saved.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

test_stack = np.column_stack((
    xgb.predict_proba(X_test)[:, 1],
    lgbm.predict_proba(X_test)[:, 1],
    dnn.predict(X_test).flatten()
))

probs = meta_model.predict_proba(test_stack)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1])
best_thresh = thresholds[f1_scores.argmax()]

roc_auc = roc_auc_score(y_test, probs)
pr_auc = average_precision_score(y_test, probs)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print("Best Threshold (F1-max):", best_thresh)

with open("/content/drive/MyDrive/eCard_Validator/models2/threshold.txt", "w") as f:
    f.write(str(best_thresh))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 990us/step
ROC-AUC: 0.9807973625578538
PR-AUC: 0.8545380829908551
Best Threshold (F1-max): 0.9999548912323737


In [ ]:
import joblib

joblib.dump(scaler, "/content/drive/MyDrive/eCard_Validator/models2/scaler.pkl")
print("Scaler saved. Training pipeline completed.")

Scaler saved. Training pipeline completed.
